In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

RAW_EXAMPLES_JSON = PROJECT_DIR / "outputs" / "mllm_example_outputs" / "260703_visual_inspection_examples.json"
WRITE_VERIFIED_JSON = False
VERIFIED_JSON = BUNDLE_DIR / "verified_answers.json"


# 03 — Extract and verify answers

MLMs often explain before giving a number. This stage makes the answer a separate, inspectable field: first extract the number on the final-answer line, then verify that the extracted literal actually appears in the response. The fallback mirrors the source logic for ranges such as `2 or 3`; missing digits remain visible as failures.


In [ ]:
import pandas as pd


def first_integer(text: str) -> int | None:
    match = re.search(r"\d+", str(text))
    return int(match.group(0)) if match else None


def final_answer_number(text: str) -> int | None:
    matches = re.findall(r"final answer\s*:\s*([^\n\r]+)", str(text), flags=re.IGNORECASE)
    return first_integer(matches[-1]) if matches else None


def literal_answer_present(text: str, answer: int | None) -> bool:
    return answer is not None and re.search(rf"(?<!\d){answer}(?!\d)", str(text)) is not None


def extract_answer(reasoning: str) -> tuple[int | None, str, bool]:
    direct = final_answer_number(reasoning)
    if direct is not None:
        return direct, "final_answer_line", True
    fallback = first_integer(reasoning)
    if fallback is not None:
        return fallback, "first_integer_fallback", True
    return None, "no_integer", False


def strict_json_value(text: str) -> int:
    value = json.loads(text)
    if set(value) != {"value"} or isinstance(value["value"], bool) or not isinstance(value["value"], int) or not 0 <= value["value"] <= MAX_CHOICE:
        raise ValueError("expected JSON object {value: integer 0..300}")
    return value["value"]


def load_reasoning_rows() -> pd.DataFrame:
    if RAW_EXAMPLES_JSON.is_file():
        payload = json.loads(RAW_EXAMPLES_JSON.read_text())
        frame = pd.DataFrame(payload.get("outputs", []))
        if not frame.empty:
            return frame
    frame = pd.read_csv(OPEN_SOURCE_CSV).head(200)
    frame["reasoning"] = frame["final_value"].map(lambda value: f"Final answer: {int(value)}.")
    frame["class_key"] = frame["dataset"] + "/" + frame["task"]
    frame["tile_index"] = frame["tileid"]
    frame["gt_count"] = frame["ground_truth_label"]
    return frame


rows = load_reasoning_rows()


## Apply extraction and literal verification

The strict JSON contract is the recovery format used by the proprietary pipeline. It is shown here as a validator, not assumed: ordinary free-form responses go through the final-answer extractor, while a recovery response can be checked with `strict_json_value`.


In [ ]:
if rows.empty:
    raise RuntimeError("No reasoning rows found. Point RAW_EXAMPLES_JSON or OPEN_SOURCE_CSV at the source data in the parameter cell.")
extracted = rows["reasoning"].fillna("").map(extract_answer)
rows[["extracted_answer", "extraction_method", "extraction_ok"]] = pd.DataFrame(extracted.tolist(), index=rows.index)
rows["literal_verified"] = [literal_answer_present(text, answer) for text, answer in zip(rows["reasoning"].fillna(""), rows["extracted_answer"])]
rows["verification_status"] = rows["literal_verified"].map({True: "verified", False: "failed"})
summary = rows.groupby("extraction_method").size().rename("rows").reset_index()
display(summary)
display(rows[["class_key", "tile_index", "reasoning", "extracted_answer", "extraction_method", "verification_status"]].head(8))
if WRITE_VERIFIED_JSON:
    VERIFIED_JSON.write_text(json.dumps(rows.to_dict(orient="records"), indent=2) + "\n")


## Visualize extraction failures and the count relationship

A failure count is more informative when paired with examples. The first panel shows how answers were obtained; the second makes failed literal verification obvious on the prediction-versus-truth plane.


In [ ]:
from mu.visv2.figure import Figure

fig = Figure(n_rows=1, n_cols=2, figsize=(12, 4.5), fontfamily="sans-serif", fontsize=11, panel_padding=(0.3, 0.2))
left, right = fig.panels
method_counts = rows["extraction_method"].value_counts()
left.p.bar(method_counts.index.tolist(), method_counts.values.tolist(), color=["#2563eb", "#f59e0b", "#dc2626"][:len(method_counts)])
left.add_xticklabel_rotation(25)
left.set_yaxislabel("rows")
left.set_title("How answers were extracted")
valid = rows[rows["extracted_answer"].notna()].copy()
colors = valid["literal_verified"].map({True: "#16a34a", False: "#dc2626"})
right.p.scatter(valid["gt_count"], valid["extracted_answer"], c=colors, s=18, alpha=0.5)
maximum = max(valid["gt_count"].max(), valid["extracted_answer"].max())
right.p.plot([0, maximum], [0, maximum], "--", color="#334155")
right.set_xaxislabel("true count")
right.set_yaxislabel("extracted count")
right.set_title("Verified (green) versus failed (red)")
fig.show()


If a model gives a range or a prose-only answer, preserve the original reasoning and the failure flag. Do not silently overwrite it with a guessed count; the verification plot is meant to expose exactly those cases.
